## Hexner demo

* The game: This is a two-player zero-sum differential game with incomplete information. ***

* The players: Following ***, P1 plays a $I$-atomic mixed strategy and P2 plays a pure strategy.

* The solver: Since the minimax problem is nonconvex-nonconcave, we use DS-GDA. Current solution is sensitive to the choice of policy network architecture and learning rate.

### TODO

* Check if dsgda_solver.py actually implements the right algorithm.

In [ ]:
"""
hexner_demo.ipynb  (as plain .py for the canvas)
================================================
Minimal notebook‑style script that
  • imports the reusable game & solver classes,
  • instantiates a HexnerGame,
  • configures and runs the DS‑GDA solver with periodic visualisation.

You can `%%bash` convert this file to a real notebook if desired:
    jupyter nbconvert --to notebook --execute hexner_demo.ipynb --output demo_run.ipynb
"""
# %% [markdown]
# # Hexner – DS‑GDA training demo
#
# This cell imports the modular game and solver classes you created in
# `hexner_game.py` and `dsgda_solver.py`, instantiates them with default
# hyper‑parameters, and runs a short DS‑GDA training loop printing the
# game value every 100 iterations together with an inline trajectory
# animation.

# %%
import importlib, dsgda_solver, player, game
from IPython.display import display
import torch
from torch import Tensor

import numpy as np
import random
SEED       = 1
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [ ]:
# ---------------------------------------------------------------------
# 1. Instantiate the game (Hexner realisation) ------------------------
# ---------------------------------------------------------------------
importlib.reload(game)
from game import HexnerGame

game_spec = {
    'tau': 0.10, 
    'T': 1.0,
    'n_types': 2,
    'BOX_POS': 2.0,
    'BOX_VEL': 30.0,
    'BOX_ACC': 8.0,
    'Z_TARGETS': torch.tensor([[0.0,  1.0, 0.0, 0.0],
                              [0.0, -1.0, 0.0, 0.0]]),
    'Kmat': torch.diag(torch.tensor([1., 1., 0., 0.])),
    'R1':   torch.diag(torch.tensor([0.05, 0.025])),
    'R2':   torch.diag(torch.tensor([0.05, 0.100])),
    'device': torch.device('cuda')
}

batch_size = 1
game = HexnerGame(game_spec, batch_size)

In [ ]:
# ---------------------------------------------------------------------
# 2. Instantiate the players ------------------------------------
# ---------------------------------------------------------------------
importlib.reload(player)
# from player import CAMS_INFORMED, BR

player_spec = {
    'hidden': 32,       # policy network width
    'temperature': 1.0  # logit temperature for mixed strategy: higher = less entropy
}

# p1 = CAMS_INFORMED(game, player_spec)
# p2 = BR(game, player_spec)

from player import P1ExplicitStrategy, BR
p1 = P1ExplicitStrategy(game, init_scale=1e-1, temperature=1.0)
p2 = BR(game, player_spec)

In [ ]:
# ---------------------------------------------------------------------
# 3. Instantiate the DS‑GDA solver ------------------------------------
# ---------------------------------------------------------------------
importlib.reload(dsgda_solver)
from dsgda_solver import DSGDASolver

solver_spec = {
    'lr_p1': 1e-1,
    'lr_p2': 1e-1,
    'momentum': 0.6,
    'C2_p1': 10.0,
    'C2_p2': 10.0,
}

solver = DSGDASolver(game, p1, p2, solver_spec, prune=True)

In [ ]:
# ---------------------------------------------------------------------
# 4. Training loop  ----------------------------------------------------
# ---------------------------------------------------------------------
import matplotlib.pyplot as plt                        # ← add this
from tree_viz import build_tree, draw_tree               # new
from IPython.display import display, Image               # already imported for traj_html
import tempfile, os


EPOCHS     = 100_000      # number of DSGDA iterations
VIS_EVERY  = 1000        # visualize solution frequency

for epoch in range(EPOCHS):
    stats = solver.step()

    if epoch % VIS_EVERY == 0:
        # save checkpoint **before** visualising
        ckpt_path = solver.save_checkpoint(epoch)
        print(f"[ckpt] saved {ckpt_path}")

        # ── helper: count paths given P1’s collapse mask ───────────────────────
        # active = solver._count_active_p1()
        # print(f"#P1 active parameters: {active}")

        print(f"[{epoch:04d}] L={stats['L']:+.4f} "
                f"nP1={stats['n_p1_active']} "
                f"t_loss={stats['t_loss']:.1f}ms "
                f"t_back={stats['t_backward']:.1f}ms "
                f"t_mom={stats['t_momentum']:.1f}ms "
                f"t_step={stats['t_step']:.1f}ms "
                f"t_col={stats['t_collapse']:.1f}ms "
                f"wall={stats['wall_ms']:.1f}ms")
        
        traj_html = game.visualize_episode(
            solver.p1, solver.p2, fps=5
        )
        display(traj_html)

        # 1) build the current tree (uses P1’s collapse flags automatically)
        # G = build_tree(
        #     solver.p1,               # current Player-1 strategy
        #     game.P0[0],              # prior belief vector
        #     game.I, game.K,          # roster size & horizon
        #     ent_thr=1e-3              # entropy threshold for “revealing”
        # )

        # 2) draw into a temporary PNG and show it
        # with tempfile.TemporaryDirectory() as tmp:
            # png_path = os.path.join(tmp, f"tree_epoch{epoch:04d}.png")
        # draw_tree(G, None)
        # plt.show()

print("Training complete.")